# Data Lineage with OpenLineage — Results Notebook

This notebook summarizes the measured results from the synthetic loan-risk data lineage project.

It covers batch model performance, group-level fairness metrics, downstream impact analysis, streaming pipeline observations, and limitations.

**Data note:** Only synthetic data is used. Groups A and B are experimental labels, not real demographic categories.

Run this notebook from the repository root so the relative paths resolve correctly.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
REPORTS = ROOT / "reports"
print("Working directory:", ROOT)
print("Reports directory exists:", REPORTS.exists())

## 1. Batch evaluation results

The pipeline produced `reports/batch_metrics.json`. The recorded values below are used if that file is not available.

In [ ]:
recorded_metrics = {
    "input_rows": 5000, "valid_rows": 5000, "rejected_rows": 0,
    "training_rows": 4042, "test_rows": 958,
    "test_accuracy": 0.8977035490605428,
    "actual_approval_rate_gap": 0.07792117195674919,
    "predicted_approval_rate_gap": 0.012896756191140613,
    "demographic_parity_ratio": 0.9811168990347785,
    "fairness_by_group": [
        {"synthetic_group":"A", "applications":470, "actual_approval_rate":0.6680851063829787,
         "predicted_approval_rate":0.6829787234042554, "true_positive_rate":0.6382978723404256},
        {"synthetic_group":"B", "applications":488, "actual_approval_rate":0.5901639344262295,
         "predicted_approval_rate":0.6700819672131147, "true_positive_rate":0.5655737704918032}
    ]
}

metrics_path = REPORTS / "batch_metrics.json"
if metrics_path.exists():
    batch_metrics = json.loads(metrics_path.read_text())
    print("Loaded", metrics_path)
else:
    batch_metrics = recorded_metrics
    print("Using recorded batch results embedded in this notebook.")

pd.DataFrame([
    ["Input records", batch_metrics["input_rows"]],
    ["Valid records", batch_metrics["valid_rows"]],
    ["Rejected records", batch_metrics["rejected_rows"]],
    ["Training records", batch_metrics["training_rows"]],
    ["Test records", batch_metrics["test_rows"]],
    ["Test accuracy", f'{batch_metrics["test_accuracy"]:.2%}'],
    ["Actual approval-rate gap", f'{batch_metrics["actual_approval_rate_gap"]:.2%}'],
    ["Predicted approval-rate gap", f'{batch_metrics["predicted_approval_rate_gap"]:.2%}'],
    ["Demographic parity ratio", f'{batch_metrics["demographic_parity_ratio"]:.4f}']
], columns=["Metric","Result"])

## 2. Batch model performance

The Logistic Regression model achieved approximately **89.77% test accuracy** on 958 held-out records. Accuracy alone does not establish fairness or suitability for lending decisions.

In [ ]:
accuracy = batch_metrics["test_accuracy"]
plt.figure(figsize=(6,4))
plt.bar(["Test accuracy"], [accuracy])
plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.title("Batch Logistic Regression — Test Accuracy")
plt.text(0, accuracy + 0.025, f"{accuracy:.2%}", ha="center")
plt.tight_layout()
plt.show()

## 3. Group-level fairness results

The following measurements were calculated on the held-out test set.

In [ ]:
group_df = pd.DataFrame(batch_metrics["fairness_by_group"])
display(group_df)

rate_df = group_df.set_index("synthetic_group")[["actual_approval_rate","predicted_approval_rate"]] * 100
ax = rate_df.plot(kind="bar", figsize=(8,5))
ax.set_title("Actual vs Predicted Approval Rate by Synthetic Group")
ax.set_xlabel("Synthetic group")
ax.set_ylabel("Approval rate (%)")
ax.set_ylim(0,100)
ax.legend(["Actual approval rate","Predicted approval rate"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

tpr_df = group_df.set_index("synthetic_group")[["true_positive_rate"]] * 100
ax = tpr_df.plot(kind="bar", figsize=(6,4), legend=False)
ax.set_title("True Positive Rate by Synthetic Group")
ax.set_xlabel("Synthetic group")
ax.set_ylabel("True positive rate (%)")
ax.set_ylim(0,100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

tpr_gap = abs(group_df.iloc[0]["true_positive_rate"] - group_df.iloc[1]["true_positive_rate"])
print(f"Absolute true-positive-rate gap: {tpr_gap:.4f} ({tpr_gap:.2%})")

### Fairness interpretation

- Actual approval rates differed by approximately **7.79 percentage points**.
- Predicted approval rates differed by approximately **1.29 percentage points**.
- The demographic parity ratio was approximately **0.9811**.
- The absolute true-positive-rate gap was approximately **7.27 percentage points**.

The relatively small predicted approval-rate gap does not mean every fairness dimension is satisfied. The true-positive-rate difference indicates that the model's ability to correctly identify approved cases differs between the synthetic groups. These results are experimental measurements on synthetic data, not a certification of fairness.

## 4. Downstream impact analysis

The impact analysis queries Marquez lineage metadata and traverses downstream dependencies from the source dataset. It writes its result to `reports/impact_analysis.json`.

In [ ]:
recorded_impact = {
    "source_dataset":"synthetic_loan_applications",
    "affected_datasets":[
        "processed_loan_applications",
        "loan_approval_model",
        "loan_predictions",
        "approval_dashboard"
    ],
    "affected_models":["loan_approval_model"],
    "affected_dashboards":["approval_dashboard"]
}

impact_path = REPORTS / "impact_analysis.json"
if impact_path.exists():
    impact = json.loads(impact_path.read_text())
    print("Loaded", impact_path)
else:
    impact = recorded_impact
    print("Using recorded impact-analysis result embedded in this notebook.")

print("Source dataset:", impact["source_dataset"])
print("Affected downstream assets:")
for asset in impact["affected_datasets"]:
    print(" -", asset)
print("Affected models:", impact["affected_models"])
print("Affected dashboards:", impact["affected_dashboards"])

pd.DataFrame({
    "Downstream asset": impact["affected_datasets"],
    "Requires investigation if source is flagged": ["Yes"] * len(impact["affected_datasets"])
})

### Impact analysis interpretation

When `synthetic_loan_applications` is flagged for investigation, the lineage graph identifies the processed dataset, trained model, predictions, and approval dashboard downstream of it.

This supports traceability and helps teams determine which assets may need review. The implementation identifies dependencies; it does **not** automatically retrain, disable, or modify affected models.

## 5. Streaming pipeline observations

The Kafka–Flink streaming pipeline was executed successfully.

- Kafka topic: `loan_application_events`
- Flink job: `Loan Risk Streaming Pipeline`
- Processing: event validation followed by one-minute tumbling processing-time windows
- Metrics: per-group sample count, approval rate, true positive rate, demographic parity difference and ratio, and equal opportunity difference
- Lineage input: `loan_application_events`
- Lineage output: `streaming_fairness_metrics`

The Flink TaskManager logs showed validated events and calculated fairness metrics. The streaming job was visible in Marquez, and its run was subsequently marked **COMPLETED** after the job was cancelled and a completion event was emitted.

Streaming values varied across windows as event counts and group distributions changed. Individual windows are not treated as a final fairness assessment because small or changing samples can produce unstable estimates.

No overall streaming fairness score is claimed in this notebook.

## 6. Testing and validation

The automated test suite completed with **17 tests passed**.

Tests covered:
- Fairness metric calculations and invalid inputs.
- Downstream impact graph traversal and edge cases.
- Presence of expected batch and streaming lineage components.

The runtime lineage for the Spark batch pipeline and Kafka–Flink streaming pipeline was also checked separately through the Marquez interface.

The automated lineage coverage tests check expected code structure and files; they do not replace end-to-end runtime validation.

## 7. Overall findings and limitations

### Findings

1. The batch Logistic Regression model achieved 89.77% test accuracy.
2. Group-level fairness measurements exposed differences that accuracy alone would not show.
3. The streaming pipeline calculated fairness indicators over incoming events.
4. OpenLineage and Marquez recorded and visualized dependencies across the workflows.
5. The impact analysis identified four downstream assets associated with the source dataset.

### Limitations

- The data is synthetic and does not represent real loan applicants.
- Evaluation uses one held-out batch test split.
- Groups A and B are experimental labels.
- Fairness metrics represent only selected dimensions of fairness.
- Streaming metrics are sensitive to window size and sample distribution.
- Lineage-based impact identification does not automatically trigger remediation.

### Conclusion

The experiment demonstrates how Big Data processing, lineage tracking, and fairness evaluation can be combined in a reproducible workflow. It also demonstrates why model performance should be considered alongside fairness indicators and data dependency information.